In [1]:
%pip install ollama

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Using qwen2.5-coder 14b because its really good at understanding structured problems like databases and gives more consistent outputs than smaller models. It also runs locally so we get fast responses and full control

In [2]:
import ollama # type: ignore
import json
import re
from typing import List

MODEL="qwen2.5-coder:14b"

In [3]:
system_prompt="""
        You are a database requirement extractor.

        STRICT RULES:
        - Return ONLY valid JSON (no markdown, no ```json)
        - Do NOT explain anything
        - Do NOT generate SQL

        OUTPUT FORMAT:

        {
        "entities": [
            {
            "name": "",
            "attributes": [
                {"name": "", "type_hint": "", "nullable": true}
            ],
            "primary_key": "",
            "description": ""
            }
        ],
        "relationships": [
            {
            "from_entity": "",
            "to_entity": "",
            "type": "one-to-one | one-to-many | many-to-many",
            "via": "",
            "description": ""
            }
        ],
        "constraints": [],
        "assumptions": []
        }
        """

Temperature is kept low to make output more consistent and less random

In [4]:
def call_model(messages):
    response=ollama.chat(
        model=MODEL,
        messages=messages,
        options={"temperature": 0.1}
    )
    return response['message']['content']

In [5]:
def clean_json(text):
    text=text.strip()

    if text.startswith("```"):
        text=text.split("```")[1]

    return text.strip()

If JSON is invalid, we send the error back to the model and ask it to fix itself. This makes the system more reliable instead of failing immediately

In [6]:
def extract_with_feedback(user_prompt,max_retries=3):
    messages=[
        {"role":"system","content":system_prompt},
        {"role":"user","content":user_prompt}
    ]

    for attempt in range(max_retries):
        print(f"\nAttempt {attempt+1}")

        output=call_model(messages)
        cleaned=clean_json(output)

        try:
            parsed=json.loads(cleaned)
            print("Valid JSON generated!")
            return parsed

        except Exception as e:
            print("Invalid JSON. Sending feedback to model...")

            messages.append({
                "role":"assistant",
                "content":output
            })

            messages.append({
                "role":"user",
                "content":f"""
                        The previous output was invalid JSON.

                        Error:
                        {str(e)}

                        Fix the JSON.
                        Return ONLY valid JSON.
                        Do NOT add markdown or explanation.
                        """
            })

    raise ValueError("Failed to generate valid JSON after retries")

### Unit Testing 1

In [7]:
user_prompt="""
        Create a college system where:
        - students enroll in courses
        - teachers teach courses
        - each course has attendance and marks
        """

result=extract_with_feedback(user_prompt)

print("\nFinal Output:\n")
print(json.dumps(result, indent=2))


Attempt 1
Valid JSON generated!

Final Output:

{
  "entities": [
    {
      "name": "Student",
      "attributes": [
        {
          "name": "student_id",
          "type_hint": "integer",
          "nullable": false
        },
        {
          "name": "first_name",
          "type_hint": "string",
          "nullable": false
        },
        {
          "name": "last_name",
          "type_hint": "string",
          "nullable": false
        },
        {
          "name": "email",
          "type_hint": "string",
          "nullable": true
        }
      ],
      "primary_key": "student_id",
      "description": "Represents a student in the college system."
    },
    {
      "name": "Teacher",
      "attributes": [
        {
          "name": "teacher_id",
          "type_hint": "integer",
          "nullable": false
        },
        {
          "name": "first_name",
          "type_hint": "string",
          "nullable": false
        },
        {
          "name": "la

In [8]:
ALLOWED_REL_TYPES={"one-to-one","one-to-many","many-to-many"}
NAME_PATTERN=re.compile(r"^[A-Za-z][A-Za-z0-9_]*$")

In [9]:
def normalize_attribute(attr):
    if isinstance(attr,str):
        name=attr.strip()
        if not name:
            return None
        return {"name":name,"type_hint":"","nullable":True}

    if isinstance(attr,dict):
        name=str(attr.get("name","")).strip()
        if not name:
            return None
        return {
            "name":name,
            "type_hint":str(attr.get("type_hint","")).strip(),
            "nullable":bool(attr.get("nullable",True))
        }

    return None

This is the core validation engine of the pipeline.
It checks whether the extracted JSON follows correct database design rules.

What it does:
- Validates structure (entities, relationships, constraints, assumptions)
- Detects duplicate entities and attributes
- Ensures primary keys exist and are valid
- Verifies relationships reference valid entities
- Checks relationship types (1-1, 1-M, M-M)
- Normalizes the data into a clean intermediate format

This is used because LLM output is not fully reliable, so we cannot directly generate SQL from it.
This step acts like a intermeediate checker that catches errors before moving forward.
It ensures only valid and clean schema representations proceed to the next step.

In [10]:
def validate_schema_ir(data):
    errors:List[str]=[]
    warnings:List[str]=[]

    if not isinstance(data,dict):
        return ["Top level input must be a JSON object."],[],{}

    entities=data.get("entities",[])
    relationships=data.get("relationships",[])
    constraints=data.get("constraints",[])
    assumptions=data.get("assumptions",[])

    if not isinstance(constraints,list):
        warnings.append("constraints should be a list (auto converted)")
        constraints=[str(constraints)]

    if not isinstance(assumptions,list):
        warnings.append("assumptions should be a list (auto converted)")
        assumptions=[str(assumptions)]

    if not isinstance(entities,list):
        errors.append("entities must be a list")
        entities=[]

    if not isinstance(relationships,list):
        errors.append("relationships must be a list")
        relationships=[]

    normalized_entities=[]
    normalized_relationships=[]
    entity_names=set()
    seen_relationships=set()

    all_attribute_names=set()

    for i,entity in enumerate(entities):
        if not isinstance(entity,dict):
            errors.append(f"Entity at index {i} must be an object")
            continue

        name=str(entity.get("name","")).strip()
        primary_key=str(entity.get("primary_key","")).strip()
        description=str(entity.get("description","")).strip()

        if not name:
            errors.append(f"Entity at index {i} is missing name")
            continue

        if name in entity_names:
            errors.append(f"Duplicate entity name found: '{name}'")
        entity_names.add(name)

        if not NAME_PATTERN.match(name):
            warnings.append(f"Entity name '{name}' is not a clean identifier")

        raw_attrs=entity.get("attributes",[])
        if not isinstance(raw_attrs,list):
            errors.append(f"Attributes of entity '{name}' must be a list")
            raw_attrs=[]

        if len(raw_attrs)==0:
            warnings.append(f"Entity '{name}' has no attributes")

        attrs=[]
        attr_names=set()

        for j,attr in enumerate(raw_attrs):
            norm=normalize_attribute(attr)
            if not norm:
                errors.append(f"Invalid attribute in entity '{name}' at index {j}")
                continue

            attr_name=norm["name"]

            if not NAME_PATTERN.match(attr_name):
                warnings.append(f"Attribute '{attr_name}' in entity '{name}' is not clean")

            if attr_name in attr_names:
                errors.append(f"Duplicate attribute '{attr_name}' in entity '{name}'")
                continue

            attr_names.add(attr_name)
            all_attribute_names.add(attr_name)  
            attrs.append(norm)

        if not primary_key:
            errors.append(f"Entity '{name}' is missing primary_key")
        elif primary_key not in attr_names:
            errors.append(
                f"Primary key '{primary_key}' for entity '{name}' is not present in its attributes"
            )

        normalized_entities.append({
            "name":name,
            "attributes":attrs,
            "primary_key":primary_key,
            "description":description
        })

    for i,rel in enumerate(relationships):
        if not isinstance(rel,dict):
            errors.append(f"Relationship at index {i} must be an object")
            continue

        from_entity=str(rel.get("from_entity",rel.get("from",""))).strip()
        to_entity=str(rel.get("to_entity",rel.get("to",""))).strip()
        rel_type=str(rel.get("type","")).strip()
        via=str(rel.get("via","")).strip()
        description=str(rel.get("description",rel.get("reason",""))).strip()

        if not from_entity or not to_entity:
            errors.append(f"Relationship at index {i} must have from_entity and to_entity")
            continue

        if from_entity not in entity_names:
            errors.append(f"Relationship at index {i} refers to unknown from_entity '{from_entity}'")

        if to_entity not in entity_names:
            errors.append(f"Relationship at index {i} refers to unknown to_entity '{to_entity}'")

        if rel_type not in ALLOWED_REL_TYPES:
            errors.append(
                f"Relationship '{from_entity}' to '{to_entity}' has invalid type '{rel_type}'"
            )

        rel_key=(from_entity,to_entity,rel_type,via)
        if rel_key in seen_relationships:
            errors.append(f"Duplicate relationship found: {from_entity} to {to_entity} ({rel_type})")
        seen_relationships.add(rel_key)

        if rel_type=="many-to-many" and not via:
            warnings.append(
                f"Many-to-many relationship '{from_entity}' to '{to_entity}' is missing 'via'"
            )

        normalized_relationships.append({
            "from_entity":from_entity,
            "to_entity":to_entity,
            "type":rel_type,
            "via":via,
            "description":description
        })

    if not normalized_entities:
        errors.append("No valid entities found")

    if not normalized_relationships:
        warnings.append("No relationships found")

    seen_constraints=set()

    for i,constraint in enumerate(constraints):
        if not isinstance(constraint,str):
            errors.append(f"Constraint at index {i} must be a string")
            continue

        c=constraint.strip()
        if not c:
            errors.append(f"Constraint at index {i} is empty")
            continue

        if c in seen_constraints:
            warnings.append(f"Duplicate constraint found: '{c}'")
            continue
        seen_constraints.add(c)

        lower=c.lower()

        if "unique" in lower:
            if not any(attr.lower() in lower for attr in all_attribute_names):
                warnings.append(f"UNIQUE constraint may not reference valid attribute: '{c}'")

        if "not null" in lower:
            if not any(attr.lower() in lower for attr in all_attribute_names):
                warnings.append(f"NOT NULL constraint may not reference valid attribute: '{c}'")

        if "primary key" in lower:
            if not any(attr.lower() in lower for attr in all_attribute_names):
                warnings.append(f"PRIMARY KEY constraint may not reference valid attribute: '{c}'")

        if "foreign key" in lower or "references" in lower:
            if not any(ent.lower() in lower for ent in entity_names):
                warnings.append(f"FOREIGN KEY constraint may not reference valid entities: '{c}'")

    normalized={
        "entities":normalized_entities,
        "relationships":normalized_relationships,
        "constraints":constraints,
        "assumptions":assumptions
    }

    return errors,warnings,normalized

In [11]:
def print_report(errors,warnings):
    print("\nValidation Report\n")

    if warnings:
        print("Warnings:")
        for w in warnings:
            print(f"  - {w}")
    else:
        print("Warnings: none")

    if errors:
        print("\nErrors:")
        for e in errors:
            print(f"  - {e}")
    else:
        print("\nErrors: none")

    print("\nResult:","Valid" if not errors else "Invalid")

### Unit Testing 2.1

In [12]:
extracted_json=extract_with_feedback(user_prompt)

print("\nExtracted JSON:\n")
print(json.dumps(extracted_json,indent=2,ensure_ascii=False))

errors,warnings,normalized=validate_schema_ir(extracted_json)

print_report(errors,warnings)


Attempt 1
Valid JSON generated!

Extracted JSON:

{
  "entities": [
    {
      "name": "Student",
      "attributes": [
        {
          "name": "student_id",
          "type_hint": "integer",
          "nullable": false
        },
        {
          "name": "first_name",
          "type_hint": "string",
          "nullable": false
        },
        {
          "name": "last_name",
          "type_hint": "string",
          "nullable": false
        },
        {
          "name": "email",
          "type_hint": "string",
          "nullable": true
        }
      ],
      "primary_key": "student_id",
      "description": "A student enrolled in the college."
    },
    {
      "name": "Teacher",
      "attributes": [
        {
          "name": "teacher_id",
          "type_hint": "integer",
          "nullable": false
        },
        {
          "name": "first_name",
          "type_hint": "string",
          "nullable": false
        },
        {
          "name": "last_name

### Unit Testing 2.2

In [13]:
bad_test_case={
    "entities": [
        {
            "name": "Student",
            "attributes": [
                {"name": "student_id", "type_hint": "int", "nullable": False},
                {"name": "student_id", "type_hint": "int", "nullable": False},
                {"name": "email", "type_hint": "string", "nullable": False}
            ],
            "primary_key": "id",
            "description": "Student details"
        },
        {
            "name": "Student",
            "attributes": ["course_id"],
            "primary_key": "course_id",
            "description": "Duplicate entity name"
        }
    ],
    "relationships": [
        {
            "from": "Student",
            "to": "Course",
            "type": "many-to-many",
            "reason": "Course entity is missing"
        }
    ],
    "constraints": "should be a list",
    "assumptions": "should also be a list"
}

print("\nTest Input:\n")
print(json.dumps(bad_test_case,indent=2,ensure_ascii=False))

errors,warnings,_=validate_schema_ir(bad_test_case)

print_report(errors,warnings)

if errors:
    print("\nValidation failed, so the next step should not run.")


Test Input:

{
  "entities": [
    {
      "name": "Student",
      "attributes": [
        {
          "name": "student_id",
          "type_hint": "int",
          "nullable": false
        },
        {
          "name": "student_id",
          "type_hint": "int",
          "nullable": false
        },
        {
          "name": "email",
          "type_hint": "string",
          "nullable": false
        }
      ],
      "primary_key": "id",
      "description": "Student details"
    },
    {
      "name": "Student",
      "attributes": [
        "course_id"
      ],
      "primary_key": "course_id",
      "description": "Duplicate entity name"
    }
  ],
  "relationships": [
    {
      "from": "Student",
      "to": "Course",
      "type": "many-to-many",
      "reason": "Course entity is missing"
    }
  ],
  "constraints": "should be a list",
  "assumptions": "should also be a list"
}

Validation Report

Warnings:
  - constraints should be a list (auto converted)
  - assumptio

In [14]:
sql_system_prompt="""
        You are an expert database schema generator.

        Your task is to convert a validated schema into SQL CREATE TABLE statements.

        STRICT RULES:
        - Output ONLY raw SQL (no explanation, no markdown, no ```sql)
        - Do NOT include any text before or after SQL
        - Use only the entities, attributes, and relationships provided
        - Do NOT invent new columns or tables

        TABLE RULES:
        - Each entity to one CREATE TABLE
        - Use the given primary_key as PRIMARY KEY
        - Use correct SQL data types (INT, VARCHAR, DATE, BOOLEAN, etc.)
        - If type is unknown to use VARCHAR(255)

        CONSTRAINT RULES:
        - nullable=false to NOT NULL
        - Apply UNIQUE constraints if mentioned
        - Respect constraints from "constraints" list if applicable
        - Do NOT ignore constraints

        RELATIONSHIP RULES:
        - one-to-many:
            - Add foreign key on the MANY side
        - one-to-one:
            - Add foreign key with UNIQUE constraint
        - many-to-many:
            - Create a junction table
            - Use 'via' name if provided
        - Include both foreign keys and composite primary key

        FOREIGN KEY RULES:
        - Use format: <entity>_id
        - Reference correct primary keys
        - Always include FOREIGN KEY constraints

        OUTPUT:
        - Only valid SQL
        - Proper indentation
        - Each CREATE TABLE separated clearly
        """

In [15]:
def build_schema_summary(schema_json):
    lines=[]

    lines.append("ENTITIES:")
    for entity in schema_json["entities"]:
        lines.append(f"- {entity['name']}")
        lines.append(f"  Primary Key: {entity['primary_key']}")
        lines.append("  Attributes:")
        for attr in entity["attributes"]:
            lines.append(
                f"    - {attr['name']} | type={attr.get('type_hint', '')} | nullable={attr.get('nullable',True)}"
            )
        if entity.get("description"):
            lines.append(f"  Description: {entity['description']}")

    lines.append("")
    lines.append("RELATIONSHIPS:")
    for rel in schema_json["relationships"]:
        lines.append(
            f"- {rel['from_entity']} to {rel['to_entity']} | type={rel['type']} | via={rel.get('via', '')}"
        )
        if rel.get("description"):
            lines.append(f"  Reason: {rel['description']}")

    lines.append("")
    lines.append("CONSTRAINTS:")
    for c in schema_json.get("constraints", []):
        lines.append(f"- {c}")

    return "\n".join(lines)

In [16]:
def clean_sql_output(text):
    text=text.strip()

    if text.startswith("```"):
        parts=text.split("```")
        if len(parts)>=2:
            text=parts[1].strip()

    if text.lower().startswith("sql"):
        text=text[3:].strip()

    return text

In [17]:
def generate_sql_with_feedback(schema_json,max_retries=3):
    schema_summary=build_schema_summary(schema_json)

    messages=[
        {"role":"system","content":sql_system_prompt},
        {
            "role":"user",
            "content":f"""
                    Generate SQL CREATE TABLE statements from this schema:

                    {schema_summary}

                    Return ONLY SQL.
                    """
        }
    ]

    for attempt in range(max_retries):
        print(f"\nSQL Generation Attempt {attempt+1}")

        output=clean_sql_output(call_model(messages))

        if "CREATE TABLE" in output.upper():
            print("SQL generated successfully!")
            return output

        messages.append({"role":"assistant","content":output})
        messages.append({
            "role":"user",
            "content":"""
                    The output is not valid SQL.

                    Fix it and return ONLY proper SQL CREATE TABLE statements.
                    No markdown. No explanation.
                    """
        })

    raise ValueError("Failed to generate valid SQL after retries")

In [18]:
extracted=extract_with_feedback(user_prompt)

errors,warnings,normalized=validate_schema_ir(extracted)
print_report(errors,warnings)

if not errors:
    print("\nGenerated SQL:\n")
    sql_output=generate_sql_with_feedback(normalized)
    print(sql_output)
else:
    print("\nFix errors before SQL generation.")  


Attempt 1
Valid JSON generated!

Validation Report

Warnings: none

Errors: none

Result: Valid

Generated SQL:


SQL Generation Attempt 1
SQL generated successfully!
CREATE TABLE Student (
    student_id INT PRIMARY KEY,
    first_name VARCHAR(255) NOT NULL,
    last_name VARCHAR(255) NOT NULL,
    email VARCHAR(255)
);

CREATE TABLE Teacher (
    teacher_id INT PRIMARY KEY,
    first_name VARCHAR(255) NOT NULL,
    last_name VARCHAR(255) NOT NULL,
    email VARCHAR(255)
);

CREATE TABLE Course (
    course_id INT PRIMARY KEY,
    course_name VARCHAR(255) NOT NULL,
    description TEXT
);

CREATE TABLE Enrollment (
    enrollment_id INT PRIMARY KEY,
    student_id INT NOT NULL,
    course_id INT NOT NULL,
    FOREIGN KEY (student_id) REFERENCES Student(student_id),
    FOREIGN KEY (course_id) REFERENCES Course(course_id)
);

CREATE TABLE Attendance (
    attendance_id INT PRIMARY KEY,
    student_id INT NOT NULL,
    course_id INT NOT NULL,
    date DATE NOT NULL,
    present BOOLEAN

In [19]:
with open("Data/Schema.sql","w",encoding="utf-8") as f:
    f.write(sql_output)